# Download Models to Triton Repository

This notebook uses the download scripts from `scripts/download/` to download models.

**Models:**
- YOLOv8n (object detection - ONNX)
- BERT base uncased (text classification - ONNX)
- Whisper tiny (audio transcription - Python backend)
- SmolLM-135M (text generation - Python backend)

**Run cells in order.** Each model can be downloaded independently.

## 1. Setup

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# =============================================================================
# Configuration
# =============================================================================
PROJECT_ROOT = Path("/mnt").resolve()
SCRIPTS_DIR = PROJECT_ROOT / "scripts" / "download"
TRITON_REPO = PROJECT_ROOT / "triton-repo"

print(f"Project root: {PROJECT_ROOT}")
print(f"Scripts directory: {SCRIPTS_DIR}")
print(f"Triton repository: {TRITON_REPO}")

# Verify scripts exist
if not SCRIPTS_DIR.exists():
    print(f"\nERROR: Scripts directory not found at {SCRIPTS_DIR}")
    print("Make sure you have cloned the repository correctly.")
else:
    print(f"\nAvailable download scripts:")
    for script in sorted(SCRIPTS_DIR.glob("download_*.py")):
        print(f"  - {script.name}")

## 2. Download All Models

Run this cell to download all models at once.

In [ ]:
# Download all models using the download_all.py script
script_path = SCRIPTS_DIR / "download_all.py"

print(f"Running: {script_path}")
print("=" * 60)

result = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=str(PROJECT_ROOT),
)

if result.returncode == 0:
    print("\n" + "=" * 60)
    print("All models downloaded successfully!")
else:
    print("\n" + "=" * 60)
    print(f"Download failed with exit code: {result.returncode}")

---
## 3. Install Model-Specific Packages

Some Python backend models require additional packages that aren't included in the base Triton image.
These packages are installed to a `packages/` folder within each model directory.

The model's `model.py` adds this folder to `sys.path` at runtime.

### 3a. Install Whisper Packages

Whisper requires the `humanize` package for formatting transcription timestamps.
This installs it to `triton-repo/models/whisper-tiny-python/packages/`.

In [ ]:
# =============================================================================
# Install humanize package for Whisper model
# =============================================================================
WHISPER_MODEL = "whisper-tiny-python"
packages_dir = TRITON_REPO / "models" / WHISPER_MODEL / "packages"

print(f"Installing packages for {WHISPER_MODEL}")
print(f"Target directory: {packages_dir}")
print("=" * 60)

# Create packages directory
packages_dir.mkdir(parents=True, exist_ok=True)

# Install humanize to the packages directory
result = subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--target", str(packages_dir),
        "--upgrade", "--quiet",
        "humanize"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"\nInstalled packages:")
    for item in sorted(packages_dir.iterdir()):
        print(f"  - {item.name}")
    print(f"\nSUCCESS: humanize installed to {packages_dir}")
else:
    print(f"\nFAILED: {result.stderr}")

---
## Individual Model Downloads (Alternative)

If you prefer to download models individually instead of using "Download All Models" above.

### Download YOLOv8n (Object Detection)

In [ ]:
script_path = SCRIPTS_DIR / "download_yolov8n.py"
result = subprocess.run([sys.executable, str(script_path)], cwd=str(PROJECT_ROOT))
print("\nYOLOv8n:", "SUCCESS" if result.returncode == 0 else "FAILED")

### Download BERT (Text Classification)

In [ ]:
script_path = SCRIPTS_DIR / "download_bert.py"
result = subprocess.run([sys.executable, str(script_path)], cwd=str(PROJECT_ROOT))
print("\nBERT:", "SUCCESS" if result.returncode == 0 else "FAILED")

### Download Whisper (Audio Transcription)

In [ ]:
script_path = SCRIPTS_DIR / "download_whisper.py"
result = subprocess.run([sys.executable, str(script_path)], cwd=str(PROJECT_ROOT))
print("\nWhisper:", "SUCCESS" if result.returncode == 0 else "FAILED")

### Download SmolLM-135M (Text Generation)

In [ ]:
script_path = SCRIPTS_DIR / "download_llm.py"
result = subprocess.run([sys.executable, str(script_path)], cwd=str(PROJECT_ROOT))
print("\nSmolLM-135M:", "SUCCESS" if result.returncode == 0 else "FAILED")

---
## 4. Summary - List Downloaded Models

In [ ]:
print(f"{'='*60}")
print("Downloaded Models Summary")
print(f"{'='*60}\n")

models_dir = TRITON_REPO / "models"
weights_dir = TRITON_REPO / "weights"

print(f"Triton Repository: {TRITON_REPO}\n")

# List models
print("Models:")
if models_dir.exists():
    for model_path in sorted(models_dir.iterdir()):
        if model_path.is_dir():
            config_exists = (model_path / "config.pbtxt").exists()
            version_dir = model_path / "1"
            has_onnx = (version_dir / "model.onnx").exists()
            has_model_py = (version_dir / "model.py").exists()
            has_packages = (model_path / "packages").exists()
            
            model_type = "ONNX" if has_onnx else "Python" if has_model_py else "Unknown"
            packages_info = " [+packages]" if has_packages else ""
            config_info = "OK" if config_exists else "X"
            
            print(f"  [{config_info}] {model_path.name} ({model_type}){packages_info}")
else:
    print("  (none - models directory not found)")

# List weights
print("\nWeights:")
if weights_dir.exists():
    for weights_path in sorted(weights_dir.iterdir()):
        if weights_path.is_dir():
            size = sum(f.stat().st_size for f in weights_path.rglob("*") if f.is_file())
            size_str = f"{size / 1024 / 1024:.1f} MB"
            print(f"  {weights_path.name} ({size_str})")
else:
    print("  (none - weights directory not found)")

print(f"\nReady for deployment!")

---
## 5. Copy Models to EDV (Optional)

Copy downloaded models to the External Data Volume for deployment.

**Prerequisites:**
- EDV must be mounted in your workspace (see Step 2 in quickstart.md)
- Update `NAMESPACE` below to match your target namespace

In [ ]:
import shutil

# =============================================================================
# Configuration - Update NAMESPACE for your environment
# =============================================================================
NAMESPACE = "domino-inference-dev"
#NAMESPACE = "domino-inference-test"
#NAMESPACE = "domino-inference-prod"
EDV_PATH = Path(f"/domino/edv/{NAMESPACE}-triton-repo-pvc")

print(f"Source: {TRITON_REPO}")
print(f"Target EDV: {EDV_PATH}")

if not EDV_PATH.exists():
    print(f"\nERROR: EDV not found at {EDV_PATH}")
    print("Make sure the EDV is mounted in your workspace.")
else:
    print(f"\nEDV found. Ready to copy.")

In [ ]:
# Copy models to EDV
if EDV_PATH.exists() and TRITON_REPO.exists():
    print("Copying models to EDV...")
    
    # Copy models directory
    src_models = TRITON_REPO / "models"
    dst_models = EDV_PATH / "models"
    if src_models.exists():
        print(f"  Copying models/ ...")
        shutil.copytree(src_models, dst_models, dirs_exist_ok=True)
    
    # Copy weights directory
    src_weights = TRITON_REPO / "weights"
    dst_weights = EDV_PATH / "weights"
    if src_weights.exists():
        print(f"  Copying weights/ ...")
        shutil.copytree(src_weights, dst_weights, dirs_exist_ok=True)
    
    print("\nCopy complete!")
    print(f"\nEDV contents:")
    for item in sorted(EDV_PATH.iterdir()):
        print(f"  {item.name}/")
else:
    print("ERROR: Cannot copy - check paths above.")